<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/VIB_FixedHorizon_H180_A60_B240.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# -*- coding: utf-8 -*-
"""Chẩn đoán nguồn H180: đo lỗi ghép ID hoặc lọc ngày, không fit mô hình.

So sánh V3 (đọc ID dạng chuỗi) với cách đọc pandas mặc định trong mã gốc.
Cách thứ hai chỉ là phép kiểm tra; không dùng để sửa ID, ngày hay đặc trưng.
"""
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Fintect')
FAILED_RUN = ROOT / (
    'VIB_FIXED_HORIZON_H180_A60_B240_V3/'
    'run_20261006T034357836181Z'
)
A_FILE = ROOT / 'final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv'
CHUNKSIZE = 250_000
ID = 'CUSTOMER_NUMBER'
SOURCES = {
    'customer': (ROOT / '1.Data_Customer.csv', 'CLIENT_CREATE_DATE'),
    'transaction': (ROOT / '2.Data_MyVIB_Transaction.csv', 'TRANS_DATE'),
    'activity': (ROOT / '3.Data_MyVIB_Activity.csv', 'ACTIVITY_DATE'),
    'deposit': (ROOT / '4.Data_Deposit.csv', 'MONTH'),
}


def clean_id(series):
    s = series.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    return s.mask(s.str.upper().isin(['', 'NAN', '<NA>', 'NONE', 'NULL']), pd.NA)


def column_names(path):
    header = pd.read_csv(path, nrows=0)
    names = {str(c).strip().upper(): c for c in header.columns}
    if len(names) != len(header.columns):
        raise ValueError(f'{path.name}: tên cột trùng sau chuẩn hóa.')
    return names


def load_preserved(path):
    names = column_names(path)
    if ID not in names:
        raise ValueError(f'{path.name}: thiếu {ID}.')
    df = pd.read_csv(path, dtype={names[ID]: str}, low_memory=False)
    df.columns = df.columns.astype(str).str.strip().str.upper()
    df[ID] = clean_id(df[ID])
    return df


def diagnostic_dates(series):
    """Ngày chưa rõ dd/mm hay mm/dd được giữ NaT trong kiểm tra cửa sổ."""
    text = series.astype('string').str.strip()
    parts = text.str.extract(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?:\D|$)')
    first = pd.to_numeric(parts[0], errors='coerce')
    second = pd.to_numeric(parts[1], errors='coerce')
    ambiguous = (first.between(1, 12) & second.between(1, 12) & first.ne(second)).fillna(False)
    parsed = pd.to_datetime(series.mask(ambiguous), format='mixed', errors='coerce')
    if not pd.api.types.is_datetime64_any_dtype(parsed) or parsed.dt.tz is not None:
        raise ValueError('Nguồn ngày có timezone hoặc kiểu hỗn hợp cần kiểm tra riêng.')
    return parsed, ambiguous


def diagnose_source(name, path, date_column, original_ids, cohort, out):
    names = column_names(path)
    missing = [c for c in [ID, date_column] if c not in names]
    if missing:
        raise ValueError(f'{path.name}: thiếu {missing}; cột có: {list(names)}')
    # Tái hiện cách nhận dạng kiểu của mã gốc, không chỉ đoán bằng cách bỏ số 0.
    inferred = pd.read_csv(path, usecols=[names[ID]], low_memory=False)[names[ID]]
    inferred_dtype = str(inferred.dtype)
    original_read_ids = clean_id(inferred)
    del inferred
    h_ids = set(cohort[ID])
    starts = cohort.set_index(ID)['RELATIONSHIP_START_DATE']
    cutoffs = cohort.set_index(ID)['CUTOFF_A']
    seen = {'V3': set(), 'original_reading': set()}
    seen_original = {'V3': set(), 'original_reading': set()}
    counts = {'source': name, 'rows': 0, 'original_inferred_ID_dtype': inferred_dtype,
              'rows_ID_text_changed_by_inference': 0,
              'ambiguous_date_rows': 0, 'nonempty_unparsed_date_rows': 0}
    for mode in seen:
        for field in ['rows_H180_ID_match', 'rows_A60_window',
                      'rows_before_relationship', 'rows_at_or_after_A60',
                      'rows_H180_with_missing_or_ambiguous_time']:
            counts[f'{field}_{mode}'] = 0
    examples, changed_pairs = [], []
    offset = 0
    reader = pd.read_csv(path, usecols=[names[ID], names[date_column]],
                         dtype={names[ID]: str}, low_memory=False, chunksize=CHUNKSIZE)
    for chunk in reader:
        chunk = chunk.reset_index(drop=True)
        raw_ids = chunk[names[ID]]
        exact_ids = clean_id(raw_ids)
        old_ids = original_read_ids.iloc[offset:offset + len(chunk)].reset_index(drop=True)
        if len(old_ids) != len(chunk):
            raise ValueError('Số dòng giữa hai lần đọc khác nhau; kiểm tra tệp nguồn đang thay đổi.')
        changed = exact_ids.fillna('<MISSING>').ne(old_ids.fillna('<MISSING>'))
        counts['rows_ID_text_changed_by_inference'] += int(changed.sum())
        pairs = pd.DataFrame({'ID_as_V3': exact_ids.to_numpy(),
                              'ID_as_original_reading': old_ids.to_numpy()})
        changed_pairs.append(pairs.dropna().drop_duplicates())
        raw_dates = chunk[names[date_column]]
        dates, ambiguous = diagnostic_dates(raw_dates)
        nonempty = raw_dates.astype('string').str.strip().notna()
        counts['ambiguous_date_rows'] += int(ambiguous.sum())
        counts['nonempty_unparsed_date_rows'] += int((nonempty & ~ambiguous & dates.isna()).sum())
        for mode, keys in [('V3', exact_ids), ('original_reading', old_ids)]:
            seen_original[mode].update(keys[keys.isin(original_ids)].dropna().unique().tolist())
            matched = keys.isin(h_ids)
            seen[mode].update(keys[matched].dropna().unique().tolist())
            r, end = keys.map(starts), keys.map(cutoffs)
            counts[f'rows_H180_ID_match_{mode}'] += int(matched.sum())
            counts[f'rows_A60_window_{mode}'] += int((matched & dates.ge(r) & dates.lt(end)).sum())
            counts[f'rows_before_relationship_{mode}'] += int((matched & dates.lt(r)).sum())
            counts[f'rows_at_or_after_A60_{mode}'] += int((matched & dates.ge(end)).sum())
            counts[f'rows_H180_with_missing_or_ambiguous_time_{mode}'] += int((matched & dates.isna()).sum())
        if len(examples) < 12:
            # Một vài ví dụ có thay đổi/matching và vài dòng nguyên bản đầu tệp.
            preferred = changed | exact_ids.isin(h_ids) | old_ids.isin(h_ids)
            selected = chunk.index[preferred].tolist()
            selected = (selected + chunk.index[:3].tolist())[:12 - len(examples)]
            for i in dict.fromkeys(selected):
                examples.append({'source': name, 'raw_ID_repr': repr(raw_ids.iloc[i]),
                    'ID_as_V3': exact_ids.iloc[i], 'ID_as_original_reading': old_ids.iloc[i],
                    'raw_date_repr': repr(raw_dates.iloc[i]),
                    'parsed_date_for_diagnostic': dates.iloc[i],
                    'ambiguous_date': bool(ambiguous.iloc[i])})
        offset += len(chunk)
    if offset != len(original_read_ids):
        raise ValueError('Hai lần đọc có tổng số dòng khác nhau; chưa thể so sánh theo dòng.')
    counts['rows'] = offset
    for mode in seen:
        counts[f'n_ID_FLDC_match_{mode}'] = len(seen_original[mode])
        counts[f'n_ID_H180_match_{mode}'] = len(seen[mode])
    if changed_pairs:
        pairs = pd.concat(changed_pairs, ignore_index=True).drop_duplicates()
        distinct = pairs.groupby('ID_as_original_reading')['ID_as_V3'].nunique()
        collision_ids = distinct[distinct > 1].index
        collisions = pairs[pairs['ID_as_original_reading'].isin(collision_ids)]
        changed_only = pairs['ID_as_V3'].ne(pairs['ID_as_original_reading'])
        pairs.loc[changed_only].to_csv(out / f'{name}_ID_text_changes_CHECK_ONLY.csv', index=False)
        collisions.to_csv(out / f'{name}_ID_inference_collisions_CHECK_ONLY.csv', index=False)
        counts['n_inference_keys_with_multiple_raw_IDs'] = len(collision_ids)
    else:
        counts['n_inference_keys_with_multiple_raw_IDs'] = 0
    pd.DataFrame(examples).to_csv(out / f'{name}_raw_ID_and_date_examples.csv', index=False)
    return counts, examples


def run_diagnostic():
    if str(ROOT).startswith('/content/') and not (Path('/content/drive') / 'MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    cohort_file = FAILED_RUN / 'cohort_fixed_horizon_master.csv'
    if not cohort_file.is_file():
        raise FileNotFoundError(f'Không tìm thấy cohort của lần chạy cần chẩn đoán: {cohort_file}')
    a, cohort = load_preserved(A_FILE), load_preserved(cohort_file)
    required = [ID, 'RELATIONSHIP_START_DATE', 'CUTOFF_A']
    if any(c not in cohort.columns for c in required) or cohort[ID].duplicated().any():
        raise ValueError('Master cohort thiếu cột bắt buộc hoặc có ID trùng.')
    for col in ['RELATIONSHIP_START_DATE', 'CUTOFF_A']:
        cohort[col] = pd.to_datetime(cohort[col], format='mixed', errors='raise')
    if cohort[required].isna().any().any():
        raise ValueError('Master cohort có ID/ngày thiếu; không tự điền.')
    out = ROOT / 'VIB_H180_SOURCE_DIAGNOSTIC' / datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S%fZ')
    out.mkdir(parents=True, exist_ok=False)
    print('Chỉ chẩn đoán; không fit mô hình, không thay ID/nhãn/ngày, không ghi đè nguồn.')
    print(f'FLDC gốc: {len(a):,}; cohort H180 đang kiểm tra: {len(cohort):,}.')
    print('V3: ID đọc dạng chuỗi. original_reading: cách pandas nhận kiểu như mã gốc; CHỈ KIỂM TRA.')
    rows, examples, errors = [], [], []
    for name, (path, date_col) in SOURCES.items():
        print(f'\nĐỌC NGUỒN: {name} — {path.name}')
        try:
            row, samples = diagnose_source(name, path, date_col, set(a[ID].dropna()), cohort, out)
            rows.append(row)
            examples.extend(samples)
            print('Hoàn tất kiểm tra:', name)
        except Exception as exc:
            errors.append({'source': name, 'error': str(exc)})
            print('Cần kiểm tra nguồn:', str(exc))
    summary = pd.DataFrame(rows)
    summary.to_csv(out / 'DIAG_source_summary.csv', index=False)
    pd.DataFrame(examples).to_csv(out / 'DIAG_source_examples.csv', index=False)
    (out / 'DIAG_errors.json').write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding='utf-8')
    id_cols = ['source', 'rows', 'original_inferred_ID_dtype',
               'n_ID_FLDC_match_V3', 'n_ID_FLDC_match_original_reading',
               'n_ID_H180_match_V3', 'n_ID_H180_match_original_reading',
               'rows_ID_text_changed_by_inference', 'n_inference_keys_with_multiple_raw_IDs']
    time_cols = ['source', 'rows_H180_ID_match_V3', 'rows_A60_window_V3',
                 'rows_before_relationship_V3', 'rows_at_or_after_A60_V3',
                 'rows_H180_ID_match_original_reading', 'rows_A60_window_original_reading',
                 'ambiguous_date_rows', 'nonempty_unparsed_date_rows']
    for title, cols in [('BẢNG 1 — ĐỐI CHIẾU CÁCH ĐỌC ID', id_cols),
                        ('BẢNG 2 — ĐỐI CHIẾU CỬA SỔ A60', time_cols)]:
        print('\n' + title)
        if not summary.empty:
            try:
                from IPython.display import display
                with pd.option_context('display.max_columns', None):
                    display(summary[cols])
            except ImportError:
                print(summary[cols].to_string(index=False))
    print('\nVÍ DỤ ID VÀ NGÀY NGUYÊN GỐC')
    if examples:
        print(pd.DataFrame(examples).groupby('source', sort=False).head(3).to_string(index=False))
    print('\nĐÃ LƯU CHẨN ĐOÁN:', out)
    print('Gửi lại Bảng 1, Bảng 2 và phần ví dụ; chưa dùng original_reading để sửa dữ liệu.')
    return summary


if __name__ == '__main__':
    DIAG_H180_SOURCE_SUMMARY = run_diagnostic()


Chỉ chẩn đoán; không fit mô hình, không thay ID/nhãn/ngày, không ghi đè nguồn.
FLDC gốc: 94,453; cohort H180 đang kiểm tra: 31,149.
V3: ID đọc dạng chuỗi. original_reading: cách pandas nhận kiểu như mã gốc; CHỈ KIỂM TRA.

ĐỌC NGUỒN: customer — 1.Data_Customer.csv
Hoàn tất kiểm tra: customer

ĐỌC NGUỒN: transaction — 2.Data_MyVIB_Transaction.csv
Hoàn tất kiểm tra: transaction

ĐỌC NGUỒN: activity — 3.Data_MyVIB_Activity.csv
Hoàn tất kiểm tra: activity

ĐỌC NGUỒN: deposit — 4.Data_Deposit.csv
Hoàn tất kiểm tra: deposit

BẢNG 1 — ĐỐI CHIẾU CÁCH ĐỌC ID


,source,rows,original_inferred_ID_dtype,n_ID_FLDC_match_V3,n_ID_FLDC_match_original_reading,n_ID_H180_match_V3,n_ID_H180_match_original_reading,rows_ID_text_changed_by_inference,n_inference_keys_with_multiple_raw_IDs
0,customer,290223,int64,0,94453,0,31149,290223,0
1,transaction,1418030,int64,0,36041,0,11248,1418030,0
2,activity,16132675,int64,0,44991,0,13396,16132675,0
3,deposit,1258424,int64,0,87976,0,29037,1258424,0



BẢNG 2 — ĐỐI CHIẾU CỬA SỔ A60


,source,rows_H180_ID_match_V3,rows_A60_window_V3,rows_before_relationship_V3,rows_at_or_after_A60_V3,rows_H180_ID_match_original_reading,rows_A60_window_original_reading,ambiguous_date_rows,nonempty_unparsed_date_rows
0,customer,0,0,0,0,31149,31149,0,0
1,transaction,0,0,0,0,566703,57742,0,0
2,activity,0,0,0,0,5267318,584546,5428635,0
3,deposit,0,0,0,0,275884,43135,0,0



VÍ DỤ ID VÀ NGÀY NGUYÊN GỐC
     source raw_ID_repr ID_as_V3 ID_as_original_reading raw_date_repr parsed_date_for_diagnostic  ambiguous_date
   customer  '00639362' 00639362                 639362  '2019-04-23'                 2019-04-23           False
   customer  '00452440' 00452440                 452440  '2019-01-11'                 2019-01-11           False
   customer  '00326290' 00326290                 326290  '2019-05-27'                 2019-05-27           False
transaction  '00701619' 00701619                 701619  '2019-01-03'                 2019-01-03           False
transaction  '00701619' 00701619                 701619  '2019-07-01'                 2019-07-01           False
transaction  '00701619' 00701619                 701619  '2019-01-02'                 2019-01-02           False
   activity  '00630178' 00630178                 630178   '1/22/2019'                 2019-01-22           False
   activity  '00630178' 00630178                 630178   '1/10/201

In [9]:
# -*- coding: utf-8 -*-
"""Chẩn đoán nguồn H180: đo lỗi ghép ID hoặc lọc ngày, không fit mô hình.

So sánh V3 (đọc ID dạng chuỗi) với cách đọc pandas mặc định trong mã gốc.
Cách thứ hai chỉ là phép kiểm tra; không dùng để sửa ID, ngày hay đặc trưng.
"""
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Fintect')
FAILED_RUN = ROOT / (
    'VIB_FIXED_HORIZON_H180_A60_B240_V3/'
    'run_20261006T034357836181Z'
)
A_FILE = ROOT / 'final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv'
CHUNKSIZE = 250_000
ID = 'CUSTOMER_NUMBER'
SOURCES = {
    'customer': (ROOT / '1.Data_Customer.csv', 'CLIENT_CREATE_DATE'),
    'transaction': (ROOT / '2.Data_MyVIB_Transaction.csv', 'TRANS_DATE'),
    'activity': (ROOT / '3.Data_MyVIB_Activity.csv', 'ACTIVITY_DATE'),
    'deposit': (ROOT / '4.Data_Deposit.csv', 'MONTH'),
}


def clean_id(series):
    s = series.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    return s.mask(s.str.upper().isin(['', 'NAN', '<NA>', 'NONE', 'NULL']), pd.NA)


def column_names(path):
    header = pd.read_csv(path, nrows=0)
    names = {str(c).strip().upper(): c for c in header.columns}
    if len(names) != len(header.columns):
        raise ValueError(f'{path.name}: tên cột trùng sau chuẩn hóa.')
    return names


def load_preserved(path):
    names = column_names(path)
    if ID not in names:
        raise ValueError(f'{path.name}: thiếu {ID}.')
    df = pd.read_csv(path, dtype={names[ID]: str}, low_memory=False)
    df.columns = df.columns.astype(str).str.strip().str.upper()
    df[ID] = clean_id(df[ID])
    return df


def diagnostic_dates(series):
    """Ngày chưa rõ dd/mm hay mm/dd được giữ NaT trong kiểm tra cửa sổ."""
    text = series.astype('string').str.strip()
    parts = text.str.extract(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?:\D|$)')
    first = pd.to_numeric(parts[0], errors='coerce')
    second = pd.to_numeric(parts[1], errors='coerce')
    ambiguous = (first.between(1, 12) & second.between(1, 12) & first.ne(second)).fillna(False)
    parsed = pd.to_datetime(series.mask(ambiguous), format='mixed', errors='coerce')
    if not pd.api.types.is_datetime64_any_dtype(parsed) or parsed.dt.tz is not None:
        raise ValueError('Nguồn ngày có timezone hoặc kiểu hỗn hợp cần kiểm tra riêng.')
    return parsed, ambiguous


def diagnose_source(name, path, date_column, original_ids, cohort, out):
    names = column_names(path)
    missing = [c for c in [ID, date_column] if c not in names]
    if missing:
        raise ValueError(f'{path.name}: thiếu {missing}; cột có: {list(names)}')
    # Tái hiện cách nhận dạng kiểu của mã gốc, không chỉ đoán bằng cách bỏ số 0.
    inferred = pd.read_csv(path, usecols=[names[ID]], low_memory=False)[names[ID]]
    inferred_dtype = str(inferred.dtype)
    original_read_ids = clean_id(inferred)
    del inferred
    h_ids = set(cohort[ID])
    starts = cohort.set_index(ID)['RELATIONSHIP_START_DATE']
    cutoffs = cohort.set_index(ID)['CUTOFF_A']
    seen = {'V3': set(), 'original_reading': set()}
    seen_original = {'V3': set(), 'original_reading': set()}
    counts = {'source': name, 'rows': 0, 'original_inferred_ID_dtype': inferred_dtype,
              'rows_ID_text_changed_by_inference': 0,
              'ambiguous_date_rows': 0, 'nonempty_unparsed_date_rows': 0}
    for mode in seen:
        for field in ['rows_H180_ID_match', 'rows_A60_window',
                      'rows_before_relationship', 'rows_at_or_after_A60',
                      'rows_H180_with_missing_or_ambiguous_time']:
            counts[f'{field}_{mode}'] = 0
    examples, changed_pairs = [], []
    offset = 0
    reader = pd.read_csv(path, usecols=[names[ID], names[date_column]],
                         dtype={names[ID]: str}, low_memory=False, chunksize=CHUNKSIZE)
    for chunk in reader:
        chunk = chunk.reset_index(drop=True)
        raw_ids = chunk[names[ID]]
        exact_ids = clean_id(raw_ids)
        old_ids = original_read_ids.iloc[offset:offset + len(chunk)].reset_index(drop=True)
        if len(old_ids) != len(chunk):
            raise ValueError('Số dòng giữa hai lần đọc khác nhau; kiểm tra tệp nguồn đang thay đổi.')
        changed = exact_ids.fillna('<MISSING>').ne(old_ids.fillna('<MISSING>'))
        counts['rows_ID_text_changed_by_inference'] += int(changed.sum())
        pairs = pd.DataFrame({'ID_as_V3': exact_ids.to_numpy(),
                              'ID_as_original_reading': old_ids.to_numpy()})
        changed_pairs.append(pairs.dropna().drop_duplicates())
        raw_dates = chunk[names[date_column]]
        dates, ambiguous = diagnostic_dates(raw_dates)
        nonempty = raw_dates.astype('string').str.strip().notna()
        counts['ambiguous_date_rows'] += int(ambiguous.sum())
        counts['nonempty_unparsed_date_rows'] += int((nonempty & ~ambiguous & dates.isna()).sum())
        for mode, keys in [('V3', exact_ids), ('original_reading', old_ids)]:
            seen_original[mode].update(keys[keys.isin(original_ids)].dropna().unique().tolist())
            matched = keys.isin(h_ids)
            seen[mode].update(keys[matched].dropna().unique().tolist())
            r, end = keys.map(starts), keys.map(cutoffs)
            counts[f'rows_H180_ID_match_{mode}'] += int(matched.sum())
            counts[f'rows_A60_window_{mode}'] += int((matched & dates.ge(r) & dates.lt(end)).sum())
            counts[f'rows_before_relationship_{mode}'] += int((matched & dates.lt(r)).sum())
            counts[f'rows_at_or_after_A60_{mode}'] += int((matched & dates.ge(end)).sum())
            counts[f'rows_H180_with_missing_or_ambiguous_time_{mode}'] += int((matched & dates.isna()).sum())
        if len(examples) < 12:
            # Một vài ví dụ có thay đổi/matching và vài dòng nguyên bản đầu tệp.
            preferred = changed | exact_ids.isin(h_ids) | old_ids.isin(h_ids)
            selected = chunk.index[preferred].tolist()
            selected = (selected + chunk.index[:3].tolist())[:12 - len(examples)]
            for i in dict.fromkeys(selected):
                examples.append({'source': name, 'raw_ID_repr': repr(raw_ids.iloc[i]),
                    'ID_as_V3': exact_ids.iloc[i], 'ID_as_original_reading': old_ids.iloc[i],
                    'raw_date_repr': repr(raw_dates.iloc[i]),
                    'parsed_date_for_diagnostic': dates.iloc[i],
                    'ambiguous_date': bool(ambiguous.iloc[i])})
        offset += len(chunk)
    if offset != len(original_read_ids):
        raise ValueError('Hai lần đọc có tổng số dòng khác nhau; chưa thể so sánh theo dòng.')
    counts['rows'] = offset
    for mode in seen:
        counts[f'n_ID_FLDC_match_{mode}'] = len(seen_original[mode])
        counts[f'n_ID_H180_match_{mode}'] = len(seen[mode])
    if changed_pairs:
        pairs = pd.concat(changed_pairs, ignore_index=True).drop_duplicates()
        distinct = pairs.groupby('ID_as_original_reading')['ID_as_V3'].nunique()
        collision_ids = distinct[distinct > 1].index
        collisions = pairs[pairs['ID_as_original_reading'].isin(collision_ids)]
        changed_only = pairs['ID_as_V3'].ne(pairs['ID_as_original_reading'])
        pairs.loc[changed_only].to_csv(out / f'{name}_ID_text_changes_CHECK_ONLY.csv', index=False)
        collisions.to_csv(out / f'{name}_ID_inference_collisions_CHECK_ONLY.csv', index=False)
        counts['n_inference_keys_with_multiple_raw_IDs'] = len(collision_ids)
    else:
        counts['n_inference_keys_with_multiple_raw_IDs'] = 0
    pd.DataFrame(examples).to_csv(out / f'{name}_raw_ID_and_date_examples.csv', index=False)
    return counts, examples


def run_diagnostic():
    if str(ROOT).startswith('/content/') and not (Path('/content/drive') / 'MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    cohort_file = FAILED_RUN / 'cohort_fixed_horizon_master.csv'
    if not cohort_file.is_file():
        raise FileNotFoundError(f'Không tìm thấy cohort của lần chạy cần chẩn đoán: {cohort_file}')
    a, cohort = load_preserved(A_FILE), load_preserved(cohort_file)
    required = [ID, 'RELATIONSHIP_START_DATE', 'CUTOFF_A']
    if any(c not in cohort.columns for c in required) or cohort[ID].duplicated().any():
        raise ValueError('Master cohort thiếu cột bắt buộc hoặc có ID trùng.')
    for col in ['RELATIONSHIP_START_DATE', 'CUTOFF_A']:
        cohort[col] = pd.to_datetime(cohort[col], format='mixed', errors='raise')
    if cohort[required].isna().any().any():
        raise ValueError('Master cohort có ID/ngày thiếu; không tự điền.')
    out = ROOT / 'VIB_H180_SOURCE_DIAGNOSTIC' / datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S%fZ')
    out.mkdir(parents=True, exist_ok=False)
    print('Chỉ chẩn đoán; không fit mô hình, không thay ID/nhãn/ngày, không ghi đè nguồn.')
    print(f'FLDC gốc: {len(a):,}; cohort H180 đang kiểm tra: {len(cohort):,}.')
    print('V3: ID đọc dạng chuỗi. original_reading: cách pandas nhận kiểu như mã gốc; CHỈ KIỂM TRA.')
    rows, examples, errors = [], [], []
    for name, (path, date_col) in SOURCES.items():
        print(f'\nĐỌC NGUỒN: {name} — {path.name}')
        try:
            row, samples = diagnose_source(name, path, date_col, set(a[ID].dropna()), cohort, out)
            rows.append(row)
            examples.extend(samples)
            print('Hoàn tất kiểm tra:', name)
        except Exception as exc:
            errors.append({'source': name, 'error': str(exc)})
            print('Cần kiểm tra nguồn:', str(exc))
    summary = pd.DataFrame(rows)
    summary.to_csv(out / 'DIAG_source_summary.csv', index=False)
    pd.DataFrame(examples).to_csv(out / 'DIAG_source_examples.csv', index=False)
    (out / 'DIAG_errors.json').write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding='utf-8')
    id_cols = ['source', 'rows', 'original_inferred_ID_dtype',
               'n_ID_FLDC_match_V3', 'n_ID_FLDC_match_original_reading',
               'n_ID_H180_match_V3', 'n_ID_H180_match_original_reading',
               'rows_ID_text_changed_by_inference', 'n_inference_keys_with_multiple_raw_IDs']
    time_cols = ['source', 'rows_H180_ID_match_V3', 'rows_A60_window_V3',
                 'rows_before_relationship_V3', 'rows_at_or_after_A60_V3',
                 'rows_H180_ID_match_original_reading', 'rows_A60_window_original_reading',
                 'ambiguous_date_rows', 'nonempty_unparsed_date_rows']
    for title, cols in [('BẢNG 1 — ĐỐI CHIẾU CÁCH ĐỌC ID', id_cols),
                        ('BẢNG 2 — ĐỐI CHIẾU CỬA SỔ A60', time_cols)]:
        print('\n' + title)
        if not summary.empty:
            try:
                from IPython.display import display
                with pd.option_context('display.max_columns', None):
                    display(summary[cols])
            except ImportError:
                print(summary[cols].to_string(index=False))
    print('\nVÍ DỤ ID VÀ NGÀY NGUYÊN GỐC')
    if examples:
        print(pd.DataFrame(examples).groupby('source', sort=False).head(3).to_string(index=False))
    print('\nĐÃ LƯU CHẨN ĐOÁN:', out)
    print('Gửi lại Bảng 1, Bảng 2 và phần ví dụ; chưa dùng original_reading để sửa dữ liệu.')
    return summary


if __name__ == '__main__':
    DIAG_H180_SOURCE_SUMMARY = run_diagnostic()


Chỉ chẩn đoán; không fit mô hình, không thay ID/nhãn/ngày, không ghi đè nguồn.
FLDC gốc: 94,453; cohort H180 đang kiểm tra: 31,149.
V3: ID đọc dạng chuỗi. original_reading: cách pandas nhận kiểu như mã gốc; CHỈ KIỂM TRA.

ĐỌC NGUỒN: customer — 1.Data_Customer.csv
Hoàn tất kiểm tra: customer

ĐỌC NGUỒN: transaction — 2.Data_MyVIB_Transaction.csv
Hoàn tất kiểm tra: transaction

ĐỌC NGUỒN: activity — 3.Data_MyVIB_Activity.csv
Hoàn tất kiểm tra: activity

ĐỌC NGUỒN: deposit — 4.Data_Deposit.csv
Hoàn tất kiểm tra: deposit

BẢNG 1 — ĐỐI CHIẾU CÁCH ĐỌC ID


,source,rows,original_inferred_ID_dtype,n_ID_FLDC_match_V3,n_ID_FLDC_match_original_reading,n_ID_H180_match_V3,n_ID_H180_match_original_reading,rows_ID_text_changed_by_inference,n_inference_keys_with_multiple_raw_IDs
0,customer,290223,int64,0,94453,0,31149,290223,0
1,transaction,1418030,int64,0,36041,0,11248,1418030,0
2,activity,16132675,int64,0,44991,0,13396,16132675,0
3,deposit,1258424,int64,0,87976,0,29037,1258424,0



BẢNG 2 — ĐỐI CHIẾU CỬA SỔ A60


,source,rows_H180_ID_match_V3,rows_A60_window_V3,rows_before_relationship_V3,rows_at_or_after_A60_V3,rows_H180_ID_match_original_reading,rows_A60_window_original_reading,ambiguous_date_rows,nonempty_unparsed_date_rows
0,customer,0,0,0,0,31149,31149,0,0
1,transaction,0,0,0,0,566703,57742,0,0
2,activity,0,0,0,0,5267318,584546,5428635,0
3,deposit,0,0,0,0,275884,43135,0,0



VÍ DỤ ID VÀ NGÀY NGUYÊN GỐC
     source raw_ID_repr ID_as_V3 ID_as_original_reading raw_date_repr parsed_date_for_diagnostic  ambiguous_date
   customer  '00639362' 00639362                 639362  '2019-04-23'                 2019-04-23           False
   customer  '00452440' 00452440                 452440  '2019-01-11'                 2019-01-11           False
   customer  '00326290' 00326290                 326290  '2019-05-27'                 2019-05-27           False
transaction  '00701619' 00701619                 701619  '2019-01-03'                 2019-01-03           False
transaction  '00701619' 00701619                 701619  '2019-07-01'                 2019-07-01           False
transaction  '00701619' 00701619                 701619  '2019-01-02'                 2019-01-02           False
   activity  '00630178' 00630178                 630178   '1/22/2019'                 2019-01-22           False
   activity  '00630178' 00630178                 630178   '1/10/201

In [10]:
# -*- coding: utf-8 -*-
# Khi chạy trong một ô Colab, chuẩn bị các gói còn thiếu trước imports.
if __name__ == '__main__':
    import importlib.util
    import subprocess
    import sys
    _packages = {'numpy': 'numpy', 'pandas': 'pandas>=2.0', 'sklearn': 'scikit-learn',
                 'xgboost': 'xgboost', 'joblib': 'joblib', 'openpyxl': 'openpyxl'}
    _missing = [p for m, p in _packages.items() if importlib.util.find_spec(m) is None]
    if _missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

# -*- coding: utf-8 -*-
"""VIB: landmark 60 ngày, theo dõi H=180 ngày, A/B cùng cohort.

Chạy file này trên Colab hoặc chạy notebook cùng tên.
Kết quả nghiên cứu chỉ được tạo khi chạy trên dữ liệu VIB của tác giả.
"""

# %% 1. IMPORTS VÀ CẤU HÌNH
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
import platform
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path('/content/drive/MyDrive/Fintect')
LANDMARK_DAYS = 60
HORIZON_DAYS = 180
# Bao gồm toàn bộ ngày 31/12/2019; không chuyển MONTH về đầu tháng.
OBSERVATION_END = pd.Timestamp('2019-12-31 23:59:59.999999999')
SPLIT_SEED = 42
MODEL_SEED = 42
BOOTSTRAP_SEED = 42
TEST_SIZE = 0.20
N_BOOTSTRAP = 1000
CSV_CHUNKSIZE = 250_000
MIN_EVENTS_FOR_REVIEW = 300  # Chỉ nhắc kiểm tra; không tự đổi H.
EXPECTED_BASE_CUSTOMERS = 94_453
EXPECTED_BASE_POSITIVES = 4_374

ID = 'CUSTOMER_NUMBER'
Y_OLD = 'TARGET_60D'
Y = 'TARGET_H'
START = 'RELATIONSHIP_START_DATE'
EVENT = 'FIRST_CARD_MONTH'
CUTOFF = 'FEATURE_CUTOFF_DATE'

STATIC_FEATURES = ['CLIENT_SEX', 'EB_REGISTER_CHANNEL', 'SMS', 'VERIFY_METHOD', 'AGE']
TRANSACTION_FEATURES = ['TRANS_LV1_MODE', 'TRANS_LV2_MODE',
                        'TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']
ACTIVITY_FEATURES = ['LOGIN_PER_ACTIVE_DAY', 'INTEREST_RATE_RATIO']
DEPOSIT_FEATURES = ['COUNT_CA_ACCT', 'AVG_CA_BALANCE', 'COUNT_TD_ACCT', 'AVG_TD_BALANCE']
LENDING_FEATURES = ['COUNT_OF_LOAN', 'AVG_LOAN_AMOUNT', 'TOTAL_LOAN_AMOUNT']
# Đúng thứ tự 18 biến trong mã A/B hiện tại của tác giả.
FEATURES = STATIC_FEATURES + ACTIVITY_FEATURES + TRANSACTION_FEATURES + DEPOSIT_FEATURES + LENDING_FEATURES

FILES = {
    'A_60D': ROOT / 'final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv',
    'customer': ROOT / '1.Data_Customer.csv',
    'card': ROOT / '6.Data_Card.xlsx',
    'transaction': ROOT / '2.Data_MyVIB_Transaction.csv',
    'activity': ROOT / '3.Data_MyVIB_Activity.csv',
    'deposit': ROOT / '4.Data_Deposit.csv',
    'lending': ROOT / '5.Data_Lending.xlsx',
}
OUTPUT_PARENT = ROOT / f'VIB_FIXED_HORIZON_H{HORIZON_DAYS}_A{LANDMARK_DAYS}_B{LANDMARK_DAYS+HORIZON_DAYS}_V3'
# Không đoán dd/mm và mm/dd. ISO yyyy-mm-dd/Excel datetime không cần thay biến này.
DATE_DAYFIRST = None
# Đã kiểm tra từ log chẩn đoán của tác giả ngày 06/10/2026:
# activity có 1/22/2019; deposit có 1/31/2019 (tháng/ngày/năm).
# Nguồn khác tiếp tục dừng khi gặp định dạng mơ hồ chưa xác định.
SOURCE_DATE_DAYFIRST = {'activity': False, 'deposit': False}
CODE_REVISION = 'V3_ID_PADDING_MMDD_20261006'
EXTRA_DATE_SOURCE_FILES = []  # Thêm đường dẫn master 60D nếu lưu ngoài ROOT.
A_AUDIT_RTOL = 1e-12
A_AUDIT_ATOL = 1e-8
DYNAMIC_FEATURES = ACTIVITY_FEATURES + TRANSACTION_FEATURES + DEPOSIT_FEATURES + LENDING_FEATURES
XGB_PARAMS = {
    'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'eval_metric': 'logloss', 'objective': 'binary:logistic',
    'random_state': MODEL_SEED, 'n_jobs': -1,
}


# %% 2. HÀM ĐỌC, KIỂM TRA VÀ LƯU
def show_table(df, title):
    print('\n' + title)
    try:
        from IPython.display import display
        display(df)
    except ImportError:
        print(df.to_string(index=False))


def require_columns(df, columns, name):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f'{name}: thiếu cột {missing}; hiện có {list(df.columns)}')


def clean_id(series):
    result = series.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    result = result.mask(result.str.upper().isin(['', 'NAN', '<NA>', 'NONE', 'NULL']), pd.NA)
    # Đã xác minh: bốn CSV gốc dùng ID số có padding (00639362),
    # A/master dùng 639362, và chẩn đoán không thấy hai ID bị gộp.
    # Chuẩn hóa chính xác trên CHUỖI; không qua float hoặc ký hiệu khoa học.
    numeric = result.str.fullmatch(r'[0-9]+', na=False)
    result.loc[numeric] = result.loc[numeric].str.lstrip('0').replace('', '0')
    return result


def clean_number(series):
    return pd.to_numeric(series.astype('string').str.replace(',', '', regex=False).str.strip(), errors='coerce')


def clean_category(series):
    result = series.astype('string').str.strip().str.upper()
    return result.mask(result.isin(['', 'NAN', '<NA>', 'NONE', 'NULL']), pd.NA)


def normalize_frame(df, name):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.upper()
    if df.columns.duplicated().any():
        raise ValueError(f'{name}: tên cột trùng sau chuẩn hóa.')
    require_columns(df, [ID], name)
    df[ID] = clean_id(df[ID])
    return df.loc[df[ID].notna()].copy()


def read_table(path, columns=None):
    path = Path(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        header = pd.read_excel(path, nrows=0)
        names = {str(c).strip().upper(): c for c in header.columns}
        if ID not in names:
            raise ValueError(f'{path.name}: thiếu {ID}.')
        df = pd.read_excel(path, dtype={names[ID]: str})
    else:
        header = pd.read_csv(path, nrows=0)
        names = {str(c).strip().upper(): c for c in header.columns}
        if ID not in names:
            raise ValueError(f'{path.name}: thiếu {ID}.')
        df = pd.read_csv(path, dtype={names[ID]: str}, low_memory=False)
    df = normalize_frame(df, path.name)
    if columns is not None:
        require_columns(df, columns, path.name)
        df = df[columns].copy()
    return df


def iter_source(path, columns):
    """CSV đọc theo chunk để tránh ghép toàn bộ lịch sử vào RAM."""
    path = Path(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        yield read_table(path, columns)
        return
    header = pd.read_csv(path, nrows=0)
    raw_names = {str(c).strip().upper(): c for c in header.columns}
    missing = [c for c in columns if c not in raw_names]
    if missing:
        raise ValueError(f'{path.name}: thiếu {missing}')
    selected = [raw_names[c] for c in columns]
    for chunk in pd.read_csv(path, usecols=selected, dtype={raw_names[ID]: str},
                             chunksize=CSV_CHUNKSIZE, low_memory=False):
        yield normalize_frame(chunk, path.name)[columns].copy()


def fail_with_rows(df, out, filename, message):
    path = out / filename
    df.to_csv(path, index=False)
    raise ValueError(f'{message}\nĐã lưu danh sách kiểm tra: {path}')


def save_json(data, path):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding='utf-8')


def file_signature(path, with_hash=False):
    stat = Path(path).stat()
    info = {'path': str(path), 'bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}
    if with_hash:
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(4 * 1024 * 1024), b''):
                digest.update(block)
        info['sha256'] = digest.hexdigest()
    return info


def validate_binary(series, name):
    values = pd.to_numeric(series, errors='coerce')
    if values.isna().any() or not values.isin([0, 1]).all():
        raise ValueError(f'{name}: nhãn phải là 0 hoặc 1, không thiếu.')
    return values.astype(np.int8)



def parse_dates(df, column, out, source):
    raw = df[column]
    text = clean_category(raw)
    parts = text.str.extract(r'^(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?:\D|$)')
    first = pd.to_numeric(parts[0], errors='coerce')
    second = pd.to_numeric(parts[1], errors='coerce')
    ambiguous = (first.between(1, 12) & second.between(1, 12) & first.ne(second)).fillna(False)
    dayfirst = SOURCE_DATE_DAYFIRST.get(source, DATE_DAYFIRST)
    if dayfirst is None and ambiguous.any():
        fail_with_rows(df.loc[ambiguous, [ID, column]], out,
                       f'CHECK_{source}_{column}_ambiguous_dates.csv',
                       'Có ngày dd/mm/yyyy hoặc mm/dd/yyyy chưa xác định định dạng. '
                       'Kiểm tra nguồn trước khi đặt DATE_DAYFIRST=True/False.')
    if source in SOURCE_DATE_DAYFIRST:
        # format='mixed' có thể tự đổi sang dd/mm nếu gặp 22/1/2019.
        # Dừng khi nguồn có dấu hiệu trái quy ước đã xác nhận, không tự đảo ngày.
        short_layout = parts[0].notna()
        left_max, right_max = (31, 12) if dayfirst else (12, 31)
        inconsistent = short_layout & (
            ~first.between(1, left_max) | ~second.between(1, right_max))
        if inconsistent.any():
            fail_with_rows(df.loc[inconsistent, [ID, column]], out,
                           f'CHECK_{source}_{column}_date_layout_conflict.csv',
                           'Nguồn có ngày trái định dạng đã xác nhận. '
                           'Cần kiểm tra nguồn; không tự đảo tháng và ngày.')
    # Giữ nguyên thời điểm; activity/deposit đọc tháng/ngày theo nguồn đã xác nhận.
    parsed = pd.to_datetime(raw, errors='coerce', format='mixed', dayfirst=bool(dayfirst))
    invalid = text.notna() & parsed.isna()
    if invalid.any():
        fail_with_rows(df.loc[invalid, [ID, column]], out,
                       f'CHECK_{source}_{column}_invalid_dates.csv', 'Có giá trị ngày chưa đọc được.')
    if not pd.api.types.is_datetime64_any_dtype(parsed) or parsed.dt.tz is not None:
        raise ValueError(f'{source}/{column}: cần thống nhất kiểu ngày và timezone với dữ liệu nguồn.')
    return parsed


def parse_numbers(df, column, out, source):
    text = clean_category(df[column])
    numbers = pd.to_numeric(text.str.replace(',', '', regex=False), errors='coerce')
    invalid = text.notna() & numbers.isna()
    if invalid.any():
        fail_with_rows(df.loc[invalid, [ID, column]], out,
                       f'CHECK_{source}_{column}_invalid_numbers.csv', 'Có giá trị chưa đọc được thành số.')
    nonfinite = numbers.notna() & ~np.isfinite(numbers.to_numpy(dtype=float, na_value=np.nan))
    if nonfinite.any():
        fail_with_rows(df.loc[nonfinite, [ID, column]], out,
                       f'CHECK_{source}_{column}_nonfinite.csv', 'Có giá trị số vô hạn.')
    return numbers


def self_check_time_rules():
    r = pd.Timestamp('2019-01-01')
    mark, end = r + pd.Timedelta(days=60), r + pd.Timedelta(days=240)
    events = pd.Series([mark, mark + pd.Timedelta(nanoseconds=1), end,
                        end + pd.Timedelta(nanoseconds=1), pd.NaT])
    y = (events.notna() & events.gt(mark) & events.le(end)).astype(int)
    assert y.tolist() == [0, 1, 1, 0, 0]
    print('PASS: nhãn mở bên ngày 60, đóng bên ngày 240.')


def discover_date_sources():
    candidates = set()
    for pattern in ['target_landmark_60d*.csv', '*master*60d*.csv']:
        candidates.update(p for p in ROOT.rglob(pattern) if p.is_file())
    candidates.update(Path(p) for p in EXTRA_DATE_SOURCE_FILES if Path(p).is_file())
    candidates.discard(FILES['A_60D'])
    return sorted(candidates, key=lambda p: str(p))


def rebuild_relationship_master(a, customer, out):
    """Ghép ngày đã lưu; không lấy ngày giao dịch/snapshot đầu làm ngày tạo.

    Nguồn bổ sung chỉ được dùng sau khi đối chiếu nhãn A60 và r+60.
    Không suy ngày tạo bằng cách trừ cutoff, không sửa ID theo phỏng đoán.
    """
    require_columns(customer, [ID, 'CLIENT_CREATE_DATE'], 'customer')
    customer = customer.loc[customer[ID].isin(a[ID])].copy()
    customer['CLIENT_CREATE_DATE'] = parse_dates(customer, 'CLIENT_CREATE_DATE', out, 'customer')
    raw_stats = customer.groupby(ID, as_index=False).agg(
        RAW_CUSTOMER_ROWS=(ID, 'size'), RAW_VALID_DATES=('CLIENT_CREATE_DATE', 'count'),
        RAW_DISTINCT_DATES=('CLIENT_CREATE_DATE', 'nunique'),
        RAW_START=('CLIENT_CREATE_DATE', 'min'))
    raw_stats.to_csv(out / 'raw_customer_date_audit.csv', index=False)
    # Cùng quy tắc mã gốc: ngày tạo sớm nhất nếu customer có nhiều dòng.
    evidence = raw_stats.loc[raw_stats['RAW_START'].notna(), [ID, 'RAW_START']].rename(
        columns={'RAW_START': START})
    evidence['DATE_SOURCE'] = str(FILES['customer'])
    evidence['SOURCE_TYPE'] = 'RAW_CUSTOMER_EARLIEST_CREATE_DATE'
    evidence_parts = [evidence]
    source_audits = []
    source_signatures = []

    def register_saved(frame, name, direct_a=False):
        frame = frame.loc[frame[ID].isin(a[ID])].copy()
        date_column = START if START in frame.columns else (
            'CLIENT_CREATE_DATE' if 'CLIENT_CREATE_DATE' in frame.columns else None)
        if date_column is None:
            source_audits.append({'source': name, 'status': 'NO_DIRECT_CREATE_DATE', 'rows': len(frame)})
            return
        if frame.empty:
            source_audits.append({'source': name, 'status': 'NO_MATCHING_A_IDS', 'rows': 0})
            return
        frame[date_column] = parse_dates(frame, date_column, out, 'saved_master')
        relevant = frame.loc[frame[date_column].notna()].copy()
        # Mỗi nguồn đã xử lý phải có cùng một ngày tạo cho một ID.
        conflicts = relevant.groupby(ID)[date_column].nunique()
        conflicts = conflicts[conflicts > 1].index
        if len(conflicts):
            fail_with_rows(relevant.loc[relevant[ID].isin(conflicts)], out,
                           'CHECK_saved_date_source_duplicate_conflict.csv',
                           f'Nguồn {name}: một ID có các ngày tạo khác nhau.')
        cutoff_col = CUTOFF if CUTOFF in frame.columns else (
            'LANDMARK_DATE' if 'LANDMARK_DATE' in frame.columns else None)
        if not direct_a and (Y_OLD not in frame.columns or cutoff_col is None):
            source_audits.append({'source': name, 'status': 'UNVERIFIED_LABEL_OR_60D_CUTOFF',
                                  'rows': len(relevant)})
            return
        if cutoff_col is not None:
            relevant[cutoff_col] = parse_dates(relevant, cutoff_col, out, 'saved_master')
            right_cutoff = relevant[cutoff_col].eq(
                relevant[date_column] + pd.Timedelta(days=LANDMARK_DAYS))
        else:
            right_cutoff = pd.Series(True, index=relevant.index)
        if Y_OLD in relevant.columns:
            relevant['_SOURCE_Y'] = pd.to_numeric(relevant[Y_OLD], errors='coerce')
            original = a.set_index(ID)[Y_OLD]
            right_label = relevant['_SOURCE_Y'].eq(relevant[ID].map(original))
        else:
            right_label = pd.Series(False, index=relevant.index)
        verified = right_cutoff & right_label
        if not verified.all():
            # Không tự dùng nguồn thuộc một cohort/nhãn/cutoff khác.
            source_audits.append({'source': name, 'status': 'UNVERIFIED_60D_CONTEXT',
                                  'rows': len(relevant), 'invalid_context_rows': int((~verified).sum())})
            relevant.loc[~verified].to_csv(
                out / f'CHECK_unverified_date_source_{len(source_audits):02d}.csv', index=False)
            return
        accepted = relevant[[ID, date_column]].drop_duplicates().rename(columns={date_column: START})
        accepted['DATE_SOURCE'] = name
        accepted['SOURCE_TYPE'] = 'VERIFIED_SAVED_60D_DATE'
        evidence_parts.append(accepted)
        source_audits.append({'source': name, 'status': 'VERIFIED', 'rows': len(accepted)})

    # Nếu A đã lưu trực tiếp ngày tạo, dùng đúng giá trị đã lưu, không suy từ cutoff.
    register_saved(a, str(FILES['A_60D']), direct_a=True)
    for path in discover_date_sources():
        print('ĐỌC LẠI NGUỒN NGÀY 60D:', path)
        source_signatures.append(file_signature(path, with_hash=True))
        saved = read_table(path)
        register_saved(saved, str(path))
    pd.DataFrame(source_audits).to_csv(out / 'saved_date_source_validation.csv', index=False)
    save_json(source_signatures, out / 'saved_date_source_signatures.json')
    all_evidence = pd.concat(evidence_parts, ignore_index=True)
    all_evidence.to_csv(out / 'relationship_date_evidence.csv', index=False)
    conflicts = all_evidence.groupby(ID)[START].nunique()
    conflicts = conflicts[conflicts > 1].index
    if len(conflicts):
        fail_with_rows(all_evidence.loc[all_evidence[ID].isin(conflicts)], out,
                       'CHECK_relationship_date_source_conflict.csv',
                       'Ngày tạo khác nhau giữa nguồn customer và các nguồn 60D đã xác minh. '
                       'Cần chọn đúng nguồn của lần chạy; mã không tự chọn một ngày.')
    resolved = all_evidence.groupby(ID, as_index=False).agg(
        RELATIONSHIP_START_DATE=(START, 'first'),
        DATE_SOURCES=('DATE_SOURCE', lambda x: ' | '.join(sorted(set(x)))),
        DATE_SOURCE_TYPES=('SOURCE_TYPE', lambda x: ' | '.join(sorted(set(x)))))
    master = a[[ID, Y_OLD]].merge(resolved, on=ID, how='left', validate='one_to_one').merge(
        raw_stats, on=ID, how='left', validate='one_to_one')
    master['DATE_FOUND_IN_RAW_CUSTOMER'] = master['RAW_START'].notna()
    master['DATE_READ_FROM_VERIFIED_SAVED_SOURCE'] = master[START].notna() & master['RAW_START'].isna()
    summary = pd.DataFrame([{
        'n_original_FLDC': len(a),
        'n_date_in_raw_customer': int(master['DATE_FOUND_IN_RAW_CUSTOMER'].sum()),
        'n_date_rejoined_from_verified_saved_source': int(master['DATE_READ_FROM_VERIFIED_SAVED_SOURCE'].sum()),
        'n_still_missing_date': int(master[START].isna().sum())}])
    master.to_csv(out / 'relationship_master_rejoined.csv', index=False)
    summary.to_csv(out / 'relationship_master_join_summary.csv', index=False)
    show_table(summary, 'GHÉP LẠI NGÀY TẠO TỪ CÁC NGUỒN ĐÃ ĐỌC')
    if master[START].isna().any():
        missing = master.loc[master[START].isna()].copy()
        missing['WHY'] = np.where(missing['RAW_CUSTOMER_ROWS'].isna(),
                                  'ID_NOT_FOUND_IN_RAW_CUSTOMER', 'RAW_CREATE_DATE_EMPTY')
        fail_with_rows(missing, out, 'CHECK_missing_relationship_date_after_all_joins.csv',
                       'Đã đọc lại customer, A và các master 60D tìm được nhưng vẫn thiếu ngày tạo. '
                       'Các nguồn không cung cấp đủ ngày thật cho những ID này. '
                       'Không suy từ giao dịch/thẻ, không tự loại khách.')
    return master


def rebuild_dynamic(master, out, branch):
    out.mkdir(parents=True, exist_ok=True)
    combined, audits = master[[ID]].copy(), []
    builders = [
        ('transaction', lambda: build_transactions(FILES['transaction'], master, out)),
        ('activity', lambda: build_activity(FILES['activity'], master, out)),
        ('deposit', lambda: build_snapshot(FILES['deposit'], master, out, 'deposit', DEPOSIT_FEATURES)),
        ('lending', lambda: build_snapshot(FILES['lending'], master, out, 'lending',
                                         ['COUNT_OF_LOAN', 'AVG_LOAN_AMOUNT']))]
    for name, builder in builders:
        print(f'\nĐỌC LẠI + TẠO ĐẶC TRƯNG {branch}: {name}')
        frame, audit = builder()
        combined = combined.merge(frame, on=ID, how='left', validate='one_to_one')
        audit['branch'] = branch
        audits.append(audit)
        print(f"Dùng {audit['rows_in_window']:,} dòng; vi phạm timestamp={audit['timestamp_violations']}.")
    require_columns(combined, [ID] + DYNAMIC_FEATURES, branch)
    combined.to_csv(out / 'all_dynamic_features.csv', index=False)
    return combined, pd.DataFrame(audits)


def verify_reconstructed_A(saved, rebuilt, numeric_columns, out):
    rebuilt = rebuilt.set_index(ID).loc[saved[ID].tolist()].reset_index()
    detail, summary = [], []
    for col in DYNAMIC_FEATURES:
        if col in numeric_columns:
            original = pd.to_numeric(saved[col], errors='coerce').to_numpy(dtype=float)
            recomputed = pd.to_numeric(rebuilt[col], errors='coerce').to_numpy(dtype=float)
            matched = np.isclose(original, recomputed, rtol=A_AUDIT_RTOL, atol=A_AUDIT_ATOL, equal_nan=True)
        else:
            matched = saved[col].astype('string').fillna('<MISSING>').eq(
                rebuilt[col].astype('string').fillna('<MISSING>')).to_numpy(dtype=bool)
        summary.append({'feature': col, 'n_checked': len(saved), 'n_mismatch': int((~matched).sum())})
        if not matched.all():
            detail.append(pd.DataFrame({ID: saved.loc[~matched, ID].to_numpy(), 'feature': col,
                                       'A_saved': saved.loc[~matched, col].to_numpy(),
                                       'A_rebuilt': rebuilt.loc[~matched, col].to_numpy()}))
    table = pd.DataFrame(summary)
    table.to_csv(out / 'A60_reconstruction_comparison.csv', index=False)
    show_table(table, 'ĐỐI CHIẾU A60 ĐÃ LƯU VỚI 13 BIẾN TÁI TẠO TỪ NGUỒN')
    if detail:
        fail_with_rows(pd.concat(detail, ignore_index=True), out, 'CHECK_A60_reconstruction_mismatch.csv',
                       'A60 tái tạo chưa khớp A đã lưu. Kiểm tra phiên bản nguồn/công thức. '
                       'Chưa dùng chênh lệch AUC để kết luận về cửa sổ; không thay A gốc.')


# %% 3. COHORT VÀ NHÃN H=180
def build_cohort(a, customer, card, out):
    """Giữ FLDC gốc; xét (r+60, r+60+H], không loại sự kiện ngày 61..180."""
    require_columns(a, [ID, Y_OLD] + FEATURES, 'A 60 ngày')
    if a[ID].duplicated().any():
        fail_with_rows(a[a[ID].duplicated(False)], out, 'CHECK_duplicate_A_ID.csv', 'A có ID trùng.')
    a = a.copy()
    a[Y_OLD] = validate_binary(a[Y_OLD], Y_OLD)
    if len(a) != EXPECTED_BASE_CUSTOMERS or int(a[Y_OLD].sum()) != EXPECTED_BASE_POSITIVES:
        raise ValueError(f'Bộ A đang đọc có n={len(a):,}, positives={int(a[Y_OLD].sum()):,}; '
                         f'khác bộ FLDC đã xác nhận ({EXPECTED_BASE_CUSTOMERS:,}; '
                         f'{EXPECTED_BASE_POSITIVES:,}). Kiểm tra đúng phiên bản tệp trước khi chạy tiếp.')

    master = rebuild_relationship_master(a, customer, out)

    require_columns(card, [ID, 'MONTH', 'COUNT_CREDITCARD'], 'card')
    card = card[card[ID].isin(a[ID])][[ID, 'MONTH', 'COUNT_CREDITCARD']].copy()
    card['MONTH'] = parse_dates(card, 'MONTH', out, 'card')
    card['COUNT_CREDITCARD'] = parse_numbers(card, 'COUNT_CREDITCARD', out, 'card')
    invalid = card['MONTH'].isna() | card['COUNT_CREDITCARD'].isna() | (card['COUNT_CREDITCARD'] < 0)
    if invalid.any():
        fail_with_rows(card[invalid], out, 'CHECK_invalid_card_rows.csv',
                       'Có dòng dữ liệu thẻ chưa xác định được ngày hoặc trạng thái. Cần kiểm tra; không tự gán 0.')
    # Giữ nguyên ngày MONTH được lưu trong tệp, theo quy ước của bộ 60 ngày.
    card = card[card['MONTH'] <= OBSERVATION_END].copy()
    observed = card.groupby(ID, as_index=False).agg(
        FIRST_OBS_MONTH=('MONTH', 'min'), LAST_OBS_MONTH=('MONTH', 'max'))
    first_card = card.loc[card['COUNT_CREDITCARD'] > 0].groupby(ID, as_index=False)['MONTH'].min().rename(
        columns={'MONTH': EVENT})
    master = master.merge(observed, on=ID, how='left', validate='one_to_one').merge(
        first_card, on=ID, how='left', validate='one_to_one')
    master['LANDMARK_DATE'] = master[START] + pd.to_timedelta(LANDMARK_DAYS, unit='D')
    master['HORIZON_END_DATE'] = master['LANDMARK_DATE'] + pd.to_timedelta(HORIZON_DAYS, unit='D')
    master['TARGET_60D_FROM_CARD'] = (
        master[EVENT].notna() & (master[EVENT] > master['LANDMARK_DATE'])).astype(np.int8)
    invalid_base = (
        master['LAST_OBS_MONTH'].isna()
        | (master['LAST_OBS_MONTH'] < master['LANDMARK_DATE'])
        | (master[EVENT].notna() & (master[EVENT] <= master['LANDMARK_DATE']))
        | (master[Y_OLD] != master['TARGET_60D_FROM_CARD'])
    )
    if invalid_base.any():
        fail_with_rows(master[invalid_base], out, 'CHECK_original_cohort_or_label_mismatch.csv',
                       'Ngày/sự kiện thẻ không khớp cohort hoặc nhãn 60 ngày gốc. '
                       'Kiểm tra phiên bản dữ liệu và quy ước MONTH; không tự sửa nhãn.')

    master['ADMIN_ELIGIBLE_H'] = master['HORIZON_END_DATE'] <= OBSERVATION_END
    master[Y] = (master[EVENT].notna() & (master[EVENT] > master['LANDMARK_DATE'])
                 & (master[EVENT] <= master['HORIZON_END_DATE'])).astype(np.int8)
    master['HAS_SNAPSHOT_AT_OR_AFTER_H'] = master['LAST_OBS_MONTH'] >= master['HORIZON_END_DATE']
    master['EVENT_AFTER_H'] = master[EVENT].notna() & (master[EVENT] > master['HORIZON_END_DATE'])
    master.to_csv(out / 'original_FLDC_horizon_eligibility_audit.csv', index=False)
    cohort = master.loc[master['ADMIN_ELIGIBLE_H']].sort_values(ID, kind='stable').reset_index(drop=True)
    if len(cohort) == 0 or cohort[Y].nunique() != 2:
        raise ValueError('Cohort H hiện tại rỗng hoặc chỉ có một lớp; cần rà soát H và dữ liệu.')
    cohort['CUTOFF_A'] = cohort['LANDMARK_DATE']
    cohort['CUTOFF_B'] = cohort['HORIZON_END_DATE'].where(cohort[Y].eq(0), cohort[EVENT])
    cohort['B_WINDOW_DAYS'] = (cohort['CUTOFF_B'] - cohort[START]).dt.total_seconds() / 86400
    if not (cohort['CUTOFF_B'] > cohort['CUTOFF_A']).all():
        raise AssertionError('Cutoff B phải sau ngày 60 trong cohort này.')
    cohort.to_csv(out / 'cohort_fixed_horizon_master.csv', index=False)
    negatives = cohort[Y].eq(0)
    missing_followup = negatives & ~cohort['HAS_SNAPSHOT_AT_OR_AFTER_H'] & ~cohort['EVENT_AFTER_H']
    # Đây là audit, không tự bổ sung tiêu chí loại khách khác với kế hoạch đã nêu.
    summary = pd.DataFrame([{
        'H_days': HORIZON_DAYS, 'landmark_days': LANDMARK_DAYS,
        'base_n': len(a), 'base_positives': int(a[Y_OLD].sum()),
        'n': len(cohort), 'events': int(cohort[Y].sum()),
        'prevalence': float(cohort[Y].mean()),
        'excluded_administrative_end': int((~master['ADMIN_ELIGIBLE_H']).sum()),
        'event_after_H_now_label_0': int(cohort['EVENT_AFTER_H'].sum()),
        'negatives_without_snapshot_at_or_after_H': int((negatives & ~cohort['HAS_SNAPSHOT_AT_OR_AFTER_H']).sum()),
        'negatives_without_endpoint_snapshot_or_later_event': int(missing_followup.sum()),
    }])
    summary.to_csv(out / 'cohort_summary.csv', index=False)
    show_table(summary, 'COHORT VÀ NHÃN THEO HORIZON CỐ ĐỊNH')
    if int(cohort[Y].sum()) < MIN_EVENTS_FOR_REVIEW:
        print(f'Có dưới {MIN_EVENTS_FOR_REVIEW} sự kiện. H vẫn là {HORIZON_DAYS}; '
              'chỉ cân nhắc H=120/90 sau khi xem số sự kiện, không chọn theo AUC.')
    if missing_followup.any():
        fail_with_rows(cohort.loc[missing_followup], out, 'CHECK_negative_followup_H.csv',
                       'Có khách nhãn 0 chưa có bằng chứng quan sát tới cuối H. '
                       'Cần xác minh nguồn theo dõi trước khi báo kết quả fixed follow-up; '
                       'mã không tự loại khách hoặc coi nhãn đó đã được xác nhận.')
    return cohort, summary


# %% 4. TẠO LẠI ĐẶC TRƯNG B TỪ DỮ LIỆU GỐC
def rows_in_window(raw, master, date_col, stats):
    raw = raw.loc[raw[ID].isin(master[ID])].copy()
    stats['rows_for_cohort'] += len(raw)
    raw[date_col] = parse_dates(raw, date_col, Path(stats['_out']), stats['source'])
    stats['invalid_date_rows'] += int(raw[date_col].isna().sum())
    joined = raw.merge(master[[ID, START, CUTOFF]], on=ID, how='inner', validate='many_to_one')
    before = joined[date_col].notna() & (joined[date_col] < joined[START])
    after = joined[date_col].notna() & (joined[date_col] >= joined[CUTOFF])
    stats['rows_before_relationship'] += int(before.sum())
    stats['rows_at_or_after_cutoff'] += int(after.sum())
    valid = joined[date_col].notna() & ~before & ~after
    selected = joined.loc[valid, raw.columns].copy()
    stats['rows_in_window'] += len(selected)
    if not selected.empty:
        bounds = selected.groupby(ID)[date_col].agg(['min', 'max', 'size'])
        for customer, row in bounds.iterrows():
            old = stats['_bounds'].get(customer)
            stats['_bounds'][customer] = (
                row['min'] if old is None else min(old[0], row['min']),
                row['max'] if old is None else max(old[1], row['max']),
                int(row['size']) + (0 if old is None else old[2]),
            )
    return selected


def new_stats(source):
    return {'source': source, 'rows_for_cohort': 0, 'invalid_date_rows': 0,
            'rows_before_relationship': 0, 'rows_at_or_after_cutoff': 0,
            'rows_in_window': 0, '_bounds': {}}


def finish_audit(stats, master, out):
    bounds = stats.pop('_bounds')
    stats.pop('_out', None)
    b = pd.DataFrame([(k, *v) for k, v in bounds.items()],
                     columns=[ID, 'MIN_SOURCE_TIME', 'MAX_SOURCE_TIME', 'N_SOURCE_ROWS'])
    audit = master[[ID, START, CUTOFF]].merge(b, on=ID, how='left', validate='one_to_one')
    for col in ['MIN_SOURCE_TIME', 'MAX_SOURCE_TIME']:
        audit[col] = pd.to_datetime(audit[col])
    audit['N_SOURCE_ROWS'] = pd.to_numeric(audit['N_SOURCE_ROWS'], errors='raise').fillna(0).astype(int)
    audit['VIOLATION'] = ((audit['MIN_SOURCE_TIME'] < audit[START])
                          | (audit['MAX_SOURCE_TIME'] >= audit[CUTOFF]))
    audit.to_csv(out / f"{stats['source']}_timestamp_audit.csv", index=False)
    if audit['VIOLATION'].any():
        raise AssertionError(f"{stats['source']}: còn dữ liệu ngoài cutoff.")
    stats['customers_with_source_rows'] = len(bounds)
    stats['customers_without_source_rows'] = len(master) - len(bounds)
    stats['timestamp_violations'] = int(audit['VIOLATION'].sum())
    return stats


def reduce_mode(parts, name):
    if not parts:
        return pd.Series(dtype=object, name=name)
    counts = pd.concat(parts).groupby([ID, 'CATEGORY'], as_index=False)['N'].sum()
    # Đồng hạng: thứ tự chữ cái như get_mode_value trong mã 180 ngày của tác giả.
    winner = counts.sort_values([ID, 'N', 'CATEGORY'], ascending=[True, False, True],
                                kind='stable').drop_duplicates(ID)
    return winner.set_index(ID)['CATEGORY'].rename(name)


def build_transactions(path, master, out):
    stats = new_stats('transaction')
    stats['_out'] = str(out)
    nums, modes1, modes2 = [], [], []
    columns = [ID, 'TRANS_DATE', 'TRANS_LV1', 'TRANS_LV2', 'TRANS_AMOUNT']
    for chunk in iter_source(path, columns):
        d = rows_in_window(chunk, master, 'TRANS_DATE', stats)
        if d.empty:
            continue
        d['TRANS_AMOUNT'] = parse_numbers(d, 'TRANS_AMOUNT', out, 'transaction')
        n = d.groupby(ID)['TRANS_AMOUNT'].agg(['sum', 'count', 'max']).rename(
            columns={'sum': 'AMOUNT_SUM', 'count': 'AMOUNT_N', 'max': 'TRANS_AMOUNT_MAX'})
        nums.append(n)
        for column, container in [('TRANS_LV1', modes1), ('TRANS_LV2', modes2)]:
            cat = clean_category(d[column])
            pairs = pd.DataFrame({ID: d[ID], 'CATEGORY': cat}).dropna(subset=['CATEGORY'])
            container.append(pairs.groupby([ID, 'CATEGORY']).size().rename('N').reset_index())
    base = master[[ID]].set_index(ID)
    if nums:
        n = pd.concat(nums).groupby(level=0).agg(
            AMOUNT_SUM=('AMOUNT_SUM', 'sum'), AMOUNT_N=('AMOUNT_N', 'sum'),
            TRANS_AMOUNT_MAX=('TRANS_AMOUNT_MAX', 'max'))
        n['TRANS_AMOUNT_MEAN'] = n['AMOUNT_SUM'] / n['AMOUNT_N'].replace(0, np.nan)
        base = base.join(n[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']])
    else:
        base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']] = np.nan
    base = base.join(reduce_mode(modes1, 'TRANS_LV1_MODE')).join(reduce_mode(modes2, 'TRANS_LV2_MODE'))
    base[['TRANS_LV1_MODE', 'TRANS_LV2_MODE']] = base[['TRANS_LV1_MODE', 'TRANS_LV2_MODE']].fillna('NO_TRANSACTION')
    base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']] = base[['TRANS_AMOUNT_MAX', 'TRANS_AMOUNT_MEAN']].fillna(0)
    feature = base.reset_index()[[ID] + TRANSACTION_FEATURES]
    feature.to_csv(out / 'transaction_features.csv', index=False)
    return feature, finish_audit(stats, master, out)


def build_activity(path, master, out):
    stats, counts, days = new_stats('activity'), [], []
    stats['_out'] = str(out)
    for chunk in iter_source(path, [ID, 'ACTIVITY_DATE', 'ACTIVITY_NAME']):
        d = rows_in_window(chunk, master, 'ACTIVITY_DATE', stats)
        if d.empty:
            continue
        category = clean_category(d['ACTIVITY_NAME'])
        d['LOGIN_N'] = category.str.contains('LOGIN', na=False, regex=False).astype(int)
        d['INTEREST_N'] = category.str.contains('INTEREST_RATE', na=False, regex=False).astype(int)
        d['ACTIVITY_N'] = 1
        counts.append(d.groupby(ID)[['LOGIN_N', 'INTEREST_N', 'ACTIVITY_N']].sum())
        days.append(pd.DataFrame({ID: d[ID], 'DAY': d['ACTIVITY_DATE'].dt.normalize()}).drop_duplicates())
    base = master[[ID]].set_index(ID)
    if counts:
        c = pd.concat(counts).groupby(level=0).sum()
        active = pd.concat(days).drop_duplicates().groupby(ID).size()
        c['LOGIN_PER_ACTIVE_DAY'] = c['LOGIN_N'] / active
        c['INTEREST_RATE_RATIO'] = c['INTEREST_N'] / c['ACTIVITY_N']
        base = base.join(c[ACTIVITY_FEATURES])
    else:
        base[ACTIVITY_FEATURES] = np.nan
    feature = base.fillna(0).reset_index()[[ID] + ACTIVITY_FEATURES]
    feature.to_csv(out / 'activity_features.csv', index=False)
    return feature, finish_audit(stats, master, out)


def build_snapshot(path, master, out, source, numeric_columns):
    stats, parts = new_stats(source), []
    stats['_out'] = str(out)
    for chunk in iter_source(path, [ID, 'MONTH'] + numeric_columns):
        d = rows_in_window(chunk, master, 'MONTH', stats)
        if d.empty:
            continue
        for col in numeric_columns:
            d[col] = parse_numbers(d, col, out, source)
        parts.append(d[[ID, 'MONTH'] + numeric_columns])
    if parts:
        all_rows = pd.concat(parts, ignore_index=True)
        maximum = all_rows.groupby(ID)['MONTH'].transform('max')
        latest = all_rows.loc[all_rows['MONTH'].eq(maximum)].drop_duplicates()
        conflicts = latest[ID].duplicated(False)
        if conflicts.any():
            fail_with_rows(latest.loc[conflicts], out, f'CHECK_{source}_latest_snapshot_conflict.csv',
                           'Cùng ID/tháng cuối có các snapshot khác nhau; cần kiểm tra nguồn. '
                           'Mã không chọn một dòng theo thứ tự đọc.')
        latest = latest[[ID] + numeric_columns]
    else:
        latest = pd.DataFrame(columns=[ID] + numeric_columns)
    feature = master[[ID]].merge(latest, on=ID, how='left', validate='one_to_one')
    feature[numeric_columns] = feature[numeric_columns].astype(float).fillna(0)
    count_cols = ['COUNT_CA_ACCT', 'COUNT_TD_ACCT'] if source == 'deposit' else ['COUNT_OF_LOAN']
    invalid_count = feature[count_cols].lt(0).any(axis=1)
    if invalid_count.any():
        fail_with_rows(feature.loc[invalid_count], out, f'CHECK_{source}_negative_counts.csv',
                       'Số tài khoản/khoản vay âm; cần kiểm tra dữ liệu.')
    for col in count_cols:
        feature[col] = feature[col].round().astype(int)
    if source == 'lending':
        feature['TOTAL_LOAN_AMOUNT'] = feature['COUNT_OF_LOAN'] * feature['AVG_LOAN_AMOUNT']
    selected = DEPOSIT_FEATURES if source == 'deposit' else LENDING_FEATURES
    nonnegative = count_cols if source == 'deposit' else selected
    invalid = feature[nonnegative].lt(0).any(axis=1) | ~np.isfinite(
        feature[selected].to_numpy(dtype=float)).all(axis=1)
    if invalid.any():
        fail_with_rows(feature.loc[invalid], out, f'CHECK_{source}_numeric_values.csv',
                       'Có giá trị đếm/cho vay âm hoặc số không hữu hạn.')
    feature = feature[[ID] + selected]
    feature.to_csv(out / f'{source}_features.csv', index=False)
    return feature, finish_audit(stats, master, out)


def build_matrices(a, cohort, out):
    numeric_columns = a[FEATURES].select_dtypes(include=np.number).columns.tolist()
    saved_a = a.set_index(ID).loc[cohort[ID].tolist(), FEATURES].reset_index()
    master_a = cohort[[ID, START, 'CUTOFF_A']].rename(columns={'CUTOFF_A': CUTOFF})
    rebuilt_a, audit_a = rebuild_dynamic(master_a, out / 'A60_REBUILD_CHECK', 'A60_check')
    verify_reconstructed_A(saved_a, rebuilt_a, numeric_columns, out)
    master_b = cohort[[ID, START, 'CUTOFF_B']].rename(columns={'CUTOFF_B': CUTOFF})
    rebuilt_b, audit_b = rebuild_dynamic(master_b, out / 'B_FIXED_HORIZON', 'B')
    matrix_a = saved_a.copy()
    matrix_a.insert(1, Y, cohort[Y].to_numpy())
    matrix_b = cohort[[ID, Y]].merge(saved_a[[ID] + STATIC_FEATURES], on=ID, how='left',
                                    validate='one_to_one').merge(rebuilt_b, on=ID, how='left',
                                                               validate='one_to_one')
    matrix_b = matrix_b[[ID, Y] + FEATURES].reset_index(drop=True)
    matrix_a = matrix_a[[ID, Y] + FEATURES].reset_index(drop=True)
    for col in FEATURES:
        if col in numeric_columns:
            converted = pd.to_numeric(matrix_b[col], errors='coerce')
            invalid = matrix_b[col].notna() & converted.isna()
            if invalid.any():
                fail_with_rows(matrix_b.loc[invalid, [ID, col]], out, f'CHECK_B_dtype_{col}.csv',
                               f'{col}: kiểu B không khớp kiểu số của A.')
            matrix_b[col] = converted
        else:
            for matrix in [matrix_a, matrix_b]:
                s = matrix[col]
                matrix[col] = s.astype(str).where(s.notna(), np.nan)
    if not matrix_a[[ID, Y]].equals(matrix_b[[ID, Y]]):
        raise AssertionError('A/B khác ID, thứ tự hoặc nhãn.')
    for col in STATIC_FEATURES:
        if not matrix_a[col].equals(matrix_b[col]):
            raise AssertionError(f'Biến dùng lại {col} khác giữa A/B.')
    for branch, matrix in [('A', matrix_a), ('B', matrix_b)]:
        if np.isinf(matrix[numeric_columns].to_numpy(dtype=float)).any():
            raise ValueError(f'{branch}: có đặc trưng số vô hạn.')
        matrix.to_csv(out / f'{branch}_fixed_horizon_matrix.csv', index=False)
    all_audits = pd.concat([audit_a, audit_b], ignore_index=True)
    all_audits.to_csv(out / 'all_source_audits.csv', index=False)
    show_table(all_audits, 'AUDIT NGUỒN ĐẶC TRƯNG A/B ĐÃ GHÉP LẠI')
    return matrix_a, matrix_b, numeric_columns


# %% 5. MÔ HÌNH, METRICS VÀ BOOTSTRAP GHÉP CẶP
def make_preprocessor(numeric_columns):
    categorical = [c for c in FEATURES if c not in numeric_columns]
    onehot_args = {'handle_unknown': 'ignore'}
    if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
        onehot_args['sparse_output'] = False
    else:
        onehot_args['sparse'] = False
    transforms = []
    if numeric_columns:
        transforms.append(('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0))]),
                           numeric_columns))
    if categorical:
        transforms.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='MISSING')),
            ('onehot', OneHotEncoder(**onehot_args)),
        ]), categorical))
    return ColumnTransformer(transforms)


def metric_row(y, p, branch, n_train):
    auc = float(roc_auc_score(y, p))
    fpr, tpr, _ = roc_curve(y, p)
    brier = float(brier_score_loss(y, p))
    prevalence = float(np.mean(y))
    baseline = prevalence * (1 - prevalence)
    k = max(1, int(np.ceil(0.10 * len(y))))
    selected = np.argsort(-np.asarray(p), kind='stable')[:k]
    true_top = int(np.asarray(y)[selected].sum())
    return {'branch': branch, 'n_train': n_train, 'n_test': len(y), 'events_test': int(np.sum(y)),
            'AUC': auc, 'KS': float(np.max(tpr - fpr)), 'Brier': brier,
            'BSS': float(1 - brier / baseline), 'Precision_at_10': true_top / k,
            'Lift_at_10': (true_top / k) / prevalence, 'Capture_at_10': true_top / int(np.sum(y))}


def paired_bootstrap(y, p_a, p_b, n_bootstrap, seed):
    """Lấy lại khách holdout có hoàn lại; A/B dùng cùng index mỗi lần.

    Không fit lại mô hình trong bootstrap. Đây là CI holdout có điều kiện
    theo lần fit/split hiện tại, khác với thí nghiệm refit 20 split đã chạy.
    """
    y, p_a, p_b = map(np.asarray, [y, p_a, p_b])
    if len(y) != len(p_a) or len(y) != len(p_b) or np.unique(y).size != 2:
        raise ValueError('Bootstrap cần cùng holdout A/B và hai lớp.')
    rng, rows = np.random.default_rng(seed), []
    attempts, max_attempts = 0, max(10000, n_bootstrap * 20)
    while len(rows) < n_bootstrap and attempts < max_attempts:
        attempts += 1
        index = rng.integers(0, len(y), size=len(y))
        yy = y[index]
        if np.unique(yy).size != 2:
            continue
        a = float(roc_auc_score(yy, p_a[index]))
        b = float(roc_auc_score(yy, p_b[index]))
        ba = float(brier_score_loss(yy, p_a[index]))
        bb = float(brier_score_loss(yy, p_b[index]))
        baseline = float(np.mean(yy) * (1 - np.mean(yy)))
        rows.append((len(rows) + 1, a, b, b - a, 1 - ba / baseline, 1 - bb / baseline))
        if len(rows) % 200 == 0 or len(rows) == n_bootstrap:
            print(f'Paired bootstrap: {len(rows)}/{n_bootstrap}')
    if len(rows) != n_bootstrap:
        raise RuntimeError(f'Chỉ có {len(rows)} bootstrap hợp lệ; chưa đủ {n_bootstrap}.')
    return pd.DataFrame(rows, columns=['replicate', 'AUC_A', 'AUC_B', 'Dwindow_H', 'BSS_A', 'BSS_B'])


def fit_and_report(a, b, numeric_columns, cohort_summary, out):
    try:
        from xgboost import XGBClassifier
    except ImportError as exc:
        raise ImportError('Chưa có xgboost. Trên Colab chạy: %pip install xgboost, rồi chạy lại.') from exc
    indexes = np.arange(len(a))
    y = a[Y].to_numpy(dtype=np.int8)
    train, test = train_test_split(indexes, test_size=TEST_SIZE, stratify=y, random_state=SPLIT_SEED)
    split = a[[ID, Y]].copy()
    split['SPLIT'] = 'TRAIN'
    split.loc[test, 'SPLIT'] = 'TEST'
    split['ORDER_IN_SPLIT'] = -1
    split.loc[train, 'ORDER_IN_SPLIT'] = np.arange(len(train))
    split.loc[test, 'ORDER_IN_SPLIT'] = np.arange(len(test))
    split.to_csv(out / 'shared_H180_train_test_split.csv', index=False)
    print(f'\nTRAIN: {len(train):,}; TEST: {len(test):,}; '
          f'sự kiện train/test: {int(y[train].sum()):,}/{int(y[test].sum()):,}')
    probabilities, rows = {}, []
    for branch, matrix in [('A', a), ('B', b)]:
        print(f'\nFIT {branch}: 18 biến; XGBoost; không lấy mẫu lại')
        model = Pipeline([('preprocess', make_preprocessor(numeric_columns)),
                          ('xgb', XGBClassifier(**XGB_PARAMS))])
        model.fit(matrix.loc[train, FEATURES], y[train])
        p = model.predict_proba(matrix.loc[test, FEATURES])[:, 1]
        probabilities[branch] = p
        rows.append(metric_row(y[test], p, branch, len(train)))
        joblib.dump(model, out / f'model_{branch}_H180.joblib')
        print(f"{branch}: AUC={rows[-1]['AUC']:.6f}; BSS={rows[-1]['BSS']:.6f}; "
              f"KS={rows[-1]['KS']:.6f}")
    predictions = pd.DataFrame({ID: a.loc[test, ID].to_numpy(), Y: y[test],
                                'PROB_A': probabilities['A'], 'PROB_B': probabilities['B']})
    predictions.to_csv(out / 'paired_H180_holdout_predictions.csv', index=False)
    boot = paired_bootstrap(y[test], probabilities['A'], probabilities['B'],
                            n_bootstrap=N_BOOTSTRAP, seed=BOOTSTRAP_SEED)
    boot.to_csv(out / 'paired_H180_bootstrap_values.csv', index=False)
    metrics = pd.DataFrame(rows)
    for index, branch in enumerate(['A', 'B']):
        ci = np.quantile(boot[f'AUC_{branch}'], [0.025, 0.975], method='linear')
        metrics.loc[index, ['AUC_ci_lower', 'AUC_ci_upper']] = ci
        ci_bss = np.quantile(boot[f'BSS_{branch}'], [0.025, 0.975], method='linear')
        metrics.loc[index, ['BSS_ci_lower', 'BSS_ci_upper']] = ci_bss
    metrics.to_csv(out / 'H180_holdout_metrics.csv', index=False)
    delta = float(rows[1]['AUC'] - rows[0]['AUC'])
    lo, hi = np.quantile(boot['Dwindow_H'], [0.025, 0.975], method='linear')
    paper = pd.DataFrame([{
        'H_days': HORIZON_DAYS, 'n': len(a), 'events': int(y.sum()),
        'n_train': len(train), 'n_test': len(test), 'events_test': int(y[test].sum()),
        'AUC_A': rows[0]['AUC'], 'BSS_A': rows[0]['BSS'], 'AUC_B': rows[1]['AUC'],
        'Dwindow_H': delta, 'Dwindow_H_ci_lower': float(lo), 'Dwindow_H_ci_upper': float(hi),
        'bootstrap_valid': len(boot), 'bootstrap_seed': BOOTSTRAP_SEED,
        'negatives_without_endpoint_snapshot_or_later_event': int(
            cohort_summary.iloc[0]['negatives_without_endpoint_snapshot_or_later_event']),
    }])
    paper.to_csv(out / 'Fixed_followup_horizon_paper_row.csv', index=False)
    # Bản văn chỉ được điền từ kết quả vừa tính trên dữ liệu thực của tác giả.
    paragraph = (
        f'With a {HORIZON_DAYS}-day follow-up horizon after the {LANDMARK_DAYS}-day landmark, '
        f'the administratively eligible cohort included {len(a):,} customers and {int(y.sum()):,} events. '
        f'Branch A used the original first-{LANDMARK_DAYS}-day features; branch B used pre-event features '
        'for adopters and features before the horizon endpoint for non-adopters. '
        'Both branches reused the same five customer attributes, including age, from branch A. '
        f'On a shared stratified 80:20 split (seed {SPLIT_SEED}), '
        f'AUC_A was {rows[0]["AUC"]:.4f}, BSS_A was {rows[0]["BSS"]:.4f}, '
        f'and AUC_B was {rows[1]["AUC"]:.4f}. '
        f'The paired difference D_window^H was {delta:.4f} '
        f'(95% paired customer-bootstrap interval {lo:.4f} to {hi:.4f}; {len(boot):,} replicates). '
        'The bootstrap interval is conditional on these fitted models and this split.'
    )
    missing = int(paper.iloc[0]['negatives_without_endpoint_snapshot_or_later_event'])
    if missing:
        paragraph += (f' For {missing:,} negative-labelled customers, endpoint follow-up was not '
                      'confirmed by a snapshot at or after the horizon or a later observed card event; '
                      'administrative eligibility alone does not resolve this limitation.')
    (out / 'Fixed_followup_horizon_paragraph_EN.txt').write_text(paragraph + '\n', encoding='utf-8')
    show_table(metrics.round(6), 'KẾT QUẢ A/B TRÊN CÙNG HOLDOUT')
    show_table(paper.round(6), 'DÒNG KẾT QUẢ ĐỂ ĐIỀN VÀO BÀI BÁO')
    print('\nĐOẠN TIẾNG ANH — kiểm tra trước khi đưa vào bài:\n' + paragraph)
    print('\nĐÃ LƯU KẾT QUẢ TẠI:\n' + str(out))
    return paper


# %% 6. CHẠY TOÀN BỘ; KẾT QUẢ HIỂN THỊ NGAY DƯỚI Ô NÀY
def run_experiment():
    """Không sửa các tệp nguồn, không ghi đè kết quả A/B/C hoặc refit 20 seed."""
    if str(ROOT).startswith('/content/'):
        try:
            from google.colab import drive
        except ImportError:
            pass
        else:
            if not (Path('/content/drive') / 'MyDrive').is_dir():
                drive.mount('/content/drive', force_remount=False)
    for name, path in FILES.items():
        if not path.is_file():
            raise FileNotFoundError(f'Thiếu nguồn {name}: {path}')
    self_check_time_rules()
    run_id = datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S%fZ')
    out = OUTPUT_PARENT / run_id
    out.mkdir(parents=True, exist_ok=False)
    packages = {}
    for name in ['pandas', 'numpy', 'scikit-learn', 'xgboost', 'joblib', 'openpyxl']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = 'not installed'
    manifest = {
        'status': 'STARTED', 'run_id': run_id, 'created_utc': datetime.now(timezone.utc).isoformat(),
        'code_revision': CODE_REVISION,
        'ID_format': 'digit-only ID: remove verified zero padding on string; never convert via float',
        'ID_format_evidence': 'author diagnostic: four CSV ID columns int64 under original reading; '
                              'leading-zero raw ID examples; no inference collisions in all four files',
        'python': platform.python_version(), 'packages': packages,
        'landmark_days': LANDMARK_DAYS, 'H_days': HORIZON_DAYS,
        'observation_end': str(OBSERVATION_END),
        'date_reading_dayfirst': DATE_DAYFIRST,
        'source_date_reading_dayfirst': SOURCE_DATE_DAYFIRST,
        'date_rejoin': 'raw customer + verified direct dates in A/saved 60D masters; stop on conflict',
        'A_dynamic_audit_tolerance': {'rtol': A_AUDIT_RTOL, 'atol': A_AUDIT_ATOL},
        'A_dynamic_audit': 'rebuild 13 first-60-day features with the same builders used for B; compare to saved A',
        'event_proxy': 'min original MONTH where COUNT_CREDITCARD > 0; MONTH is not rounded',
        'cohort': 'existing FLDC A IDs, then r+60+H <= administrative observation end',
        'label': '1 iff r+60 < FIRST_CARD_MONTH <= r+60+H',
        'feature_filter': 'RELATIONSHIP_START_DATE <= source time < FEATURE_CUTOFF_DATE',
        'A': 'reuse all original 18 A60 features; change label and cohort only',
        'B': 'reuse five A60 customer attributes; rebuild 13 behavioural features',
        'age': 'reuse AGE from A60 for both branches; do not recalculate at B cutoff',
        'negative_B_cutoff': 'r+60+H', 'positive_B_cutoff': 'FIRST_CARD_MONTH',
        'features': FEATURES, 'static_features_reused': STATIC_FEATURES,
        'split_seed': SPLIT_SEED, 'test_size': TEST_SIZE,
        'split': 'new shared stratified split using TARGET_H; not the original E2 split',
        'xgb_params': XGB_PARAMS, 'resampling': 'none', 'cv': 'not run in this horizon experiment',
        'preprocessing': 'train-only constant imputation and one-hot, as in current A/B code',
        'bootstrap': 'paired nonparametric customer bootstrap on holdout; percentile 95%; fixed fits',
        'bootstrap_seed': BOOTSTRAP_SEED, 'bootstrap_replicates': N_BOOTSTRAP,
        'BSS_definition': '1 - Brier / (p_test*(1-p_test)), matching the supplied A/B code',
        'sources': {name: file_signature(path, with_hash=name in ['A_60D', 'customer', 'card'])
                    for name, path in FILES.items()},
    }
    save_json(manifest, out / 'run_manifest.json')
    print(f'H={HORIZON_DAYS}; landmark={LANDMARK_DAYS}; mốc cuối={LANDMARK_DAYS + HORIZON_DAYS} ngày')
    print('PHIÊN BẢN CODE:', CODE_REVISION)
    print('ĐÃ SỬA: ID padding của CSV; activity/deposit dùng tháng/ngày/năm đã xác nhận.')
    print('DÙNG LẠI: năm biến khách hàng từ A60, bao gồm AGE. A giữ đủ 18 biến gốc.')
    print('THƯ MỤC CHẠY MỚI:\n' + str(out))
    started = time.monotonic()
    try:
        a = read_table(FILES['A_60D'])
        customer = read_table(FILES['customer'])
        card = read_table(FILES['card'], [ID, 'MONTH', 'COUNT_CREDITCARD'])
        cohort, summary = build_cohort(a, customer, card, out)
        matrix_a, matrix_b, numeric = build_matrices(a, cohort, out)
        result = fit_and_report(matrix_a, matrix_b, numeric, summary, out)
        manifest['status'] = 'COMPLETED'
        manifest['result'] = result.iloc[0].to_dict()
        return result
    except Exception as exc:
        manifest['status'] = 'FAILED'
        manifest['error'] = str(exc)
        print('\nCHƯA HOÀN TẤT. Kiểm tra lỗi và tệp CHECK nếu có. Không dùng kết quả dở dang để điền bài.')
        raise
    finally:
        manifest['elapsed_seconds'] = time.monotonic() - started
        save_json(manifest, out / 'run_manifest.json')


if __name__ == '__main__':
    RESULTS_FIXED_HORIZON = run_experiment()


PASS: nhãn mở bên ngày 60, đóng bên ngày 240.
H=180; landmark=60; mốc cuối=240 ngày
PHIÊN BẢN CODE: V3_ID_PADDING_MMDD_20261006
ĐÃ SỬA: ID padding của CSV; activity/deposit dùng tháng/ngày/năm đã xác nhận.
DÙNG LẠI: năm biến khách hàng từ A60, bao gồm AGE. A giữ đủ 18 biến gốc.
THƯ MỤC CHẠY MỚI:
/content/drive/MyDrive/Fintect/VIB_FIXED_HORIZON_H180_A60_B240_V3/run_20261006T055822905327Z
ĐỌC LẠI NGUỒN NGÀY 60D: /content/drive/MyDrive/Fintect/target_landmark_60d.csv

GHÉP LẠI NGÀY TẠO TỪ CÁC NGUỒN ĐÃ ĐỌC


,n_original_FLDC,n_date_in_raw_customer,n_date_rejoined_from_verified_saved_source,n_still_missing_date
0,94453,94453,0,0



COHORT VÀ NHÃN THEO HORIZON CỐ ĐỊNH


,H_days,landmark_days,base_n,base_positives,n,events,prevalence,excluded_administrative_end,event_after_H_now_label_0,negatives_without_snapshot_at_or_after_H,negatives_without_endpoint_snapshot_or_later_event
0,180,60,94453,4374,31149,1454,0.046679,63304,1093,0,0



ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: transaction
Dùng 57,742 dòng; vi phạm timestamp=0.

ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: activity
Dùng 927,440 dòng; vi phạm timestamp=0.

ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: deposit
Dùng 43,135 dòng; vi phạm timestamp=0.

ĐỌC LẠI + TẠO ĐẶC TRƯNG A60_check: lending


...


TRAIN: 24,919; TEST: 6,230; sự kiện train/test: 1,163/291

FIT A: 18 biến; XGBoost; không lấy mẫu lại
A: AUC=0.891002; BSS=0.139893; KS=0.648320

FIT B: 18 biến; XGBoost; không lấy mẫu lại
B: AUC=0.906161; BSS=0.215304; KS=0.652414
Paired bootstrap: 200/1000
Paired bootstrap: 400/1000
Paired bootstrap: 600/1000
Paired bootstrap: 800/1000
Paired bootstrap: 1000/1000

KẾT QUẢ A/B TRÊN CÙNG HOLDOUT


,branch,n_train,n_test,events_test,AUC,KS,Brier,BSS,Precision_at_10,Lift_at_10,Capture_at_10,AUC_ci_lower,AUC_ci_upper,BSS_ci_lower,BSS_ci_upper
0,A,24919,6230,291,0.891002,0.648320,0.038299,0.139893,0.252006,5.395189,0.539519,0.874029,0.904233,0.102178,0.171256
1,B,24919,6230,291,0.906161,0.652414,0.034941,0.215304,0.288925,6.185567,0.618557,0.889924,0.920374,0.168533,0.256578



DÒNG KẾT QUẢ ĐỂ ĐIỀN VÀO BÀI BÁO


,H_days,n,events,n_train,n_test,events_test,AUC_A,BSS_A,AUC_B,Dwindow_H,Dwindow_H_ci_lower,Dwindow_H_ci_upper,bootstrap_valid,bootstrap_seed,negatives_without_endpoint_snapshot_or_later_event
0,180,31149,1454,24919,6230,291,0.891002,0.139893,0.906161,0.015158,0.004543,0.026398,1000,42,0



ĐOẠN TIẾNG ANH — kiểm tra trước khi đưa vào bài:
With a 180-day follow-up horizon after the 60-day landmark, the administratively eligible cohort included 31,149 customers and 1,454 events. Branch A used the original first-60-day features; branch B used pre-event features for adopters and features before the horizon endpoint for non-adopters. Both branches reused the same five customer attributes, including age, from branch A. On a shared stratified 80:20 split (seed 42), AUC_A was 0.8910, BSS_A was 0.1399, and AUC_B was 0.9062. The paired difference D_window^H was 0.0152 (95% paired customer-bootstrap interval 0.0045 to 0.0264; 1,000 replicates). The bootstrap interval is conditional on these fitted models and this split.

ĐÃ LƯU KẾT QUẢ TẠI:
/content/drive/MyDrive/Fintect/VIB_FIXED_HORIZON_H180_A60_B240_V3/run_20261006T055822905327Z
